In [1]:
import requests
import torch
import time
import psutil
import subprocess
from unidecode import unidecode
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, hamming_loss

import pandas as pd

from datasets import load_dataset

In [2]:
ds = load_dataset("Rami/multi-label-class-github-issues-text-classification")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 778 entries, 0 to 777
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     778 non-null    object
 1   labels    778 non-null    object
 2   bodyText  778 non-null    object
dtypes: object(3)
memory usage: 18.4+ KB


In [3]:
labels = ["bug", "feature", "question", "won't fix", "docs"]

test = test[test['labels'].apply(lambda cats: all(c in labels for c in cats))]
test = test[test['labels'].apply(len) > 0]
test.rename(columns={'title': 'text'}, inplace=True)
test.drop(columns=['bodyText'], inplace=True)
test.reset_index(drop=True, inplace=True)

test

,text,labels
0,Update CONTRIBUTING.md on bugfixes/features PRs,[docs]
1,How to print the metric (across all working tr...,"[question, won't fix]"
2,Model loaded from checkpoint has bad accuracy,[question]
3,Add a robots.txt to stop Google from indexing ...,[docs]
4,Multi-processing with IterableDataset Warning,[docs]
...,...,...
195,TensorBoardLogger and ModelCheckpoint are not ...,[bug]
196,imagenet_example cannot run,[bug]
197,log_gpu_memory='all'` options raise Error,[bug]
198,`overfit_pct` vs `train_percent_check` etc,"[feature, docs]"


In [4]:
mlb = MultiLabelBinarizer()
test_labels_binarized = mlb.fit_transform(test['labels'])

test_labels_df = pd.DataFrame(test_labels_binarized, columns=mlb.classes_)

test = pd.concat([test, test_labels_df], axis=1)

test.drop(columns=['labels'], inplace=True)

In [5]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_12904\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


39923712

In [6]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [7]:
def classify(text, labels):

    url = "http://localhost:11434/api/chat"

    payload = {
        "model": "llama3.2:3b",
        "messages" : [
            {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling multilabel classification tasks based on user instructions."},
            {"role": "user", "content": f"Classify the following text based on the task: Classification of github issues. Only respond with the labels that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
        ],
        "stream": False,
        "options": {
            "temperature": 0
        }
    }

    start_time = time.time()
    response = requests.post(url, json=payload)
    response_time = time.time() - start_time

    vram_usage = get_gpu_memory_usage()

    ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)

    response = response.json()
    total_time = response['total_duration'] / 1_000_000_000
    content = unidecode(response['message']['content'].lower())

    list = []

    if 'bug' in content:
        list.append('bug')
    if 'feature' in content:
        list.append('feature')
    if 'question' in content:
        list.append('question')
    if "won't fix" in content:
        list.append("won't fix")
    if 'docs' in content:
        list.append('docs')

    return list, response_time, vram_usage, ram_usage_bytes, total_time

In [9]:
# apply the classify function to the test set. create one column for each output
test[['prediction', 'response_time', 'vram_usage', 'ram_usage', 'total_time']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_12904\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


In [10]:
for label in labels:
    test[f"{label} pred"] = test.apply(lambda row: 1 if label in row['prediction'] else 0, axis=1)

test = test.drop(columns=['prediction'])

test

,text,bug,docs,feature,question,won't fix,response_time,vram_usage,ram_usage,total_time,bug pred,feature pred,question pred,won't fix pred,docs pred
2,Model loaded from checkpoint has bad accuracy,0,0,0,1,0,3.486517,4227,96.906250,1.437736,1,0,0,0,0
150,Recursive device conversion of tuple,1,0,0,0,0,2.105751,4223,97.441406,0.050380,0,1,0,0,0
199,[Pyright] Cannot instantiate abstract class,1,0,0,0,1,2.090541,4227,97.265625,0.043871,1,0,0,0,0
29,Error in example of learning rate finder docs,0,1,0,0,0,2.267083,4218,97.910156,0.225607,0,0,0,0,1
59,How was the README animation created ?,0,0,0,1,1,2.288000,4218,97.945312,0.245384,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31,How to use Lightning 1.0 without validation da...,0,0,0,1,0,2.257280,4209,98.640625,0.213884,0,1,0,0,0
22,"[docs] use `versionadded`, `versionchanged` an...",0,1,0,0,1,2.451261,4209,97.542969,0.414077,0,0,0,0,1
177,training_forward assumes input has .copy() met...,1,0,0,0,0,2.144771,4209,98.085938,0.092964,0,1,0,0,0
52,Set different lr_schedulers for different para...,0,0,0,1,0,2.304987,4209,98.089844,0.252784,0,1,0,0,0


In [11]:
y_true = test[labels].values
y_pred = test[[f"{label} pred" for label in labels]].values

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)
hamming_loss = hamming_loss(y_true, y_pred)
print('Hamming loss: %f' % hamming_loss)

Accuracy: 0.188034
F1 score: 0.267267
Precision: 0.678594
Recall: 0.256944
Hamming loss: 0.335043


In [12]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 2.2970661770584235
Average VRAM usage: 4212.991452991453
Average RAM usage: 98.0772235576923
Average total time: 0.25273684529914525


In [13]:
# save results to txt
with open('results/gemma_ZS_multilabel2.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Hamming loss: {hamming_loss}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')